# Subpopulation Plots (cell-by-cell)

Same figures and macros as `plot_subpop_results.py` / `compute_ppds.py`, but as
individually re-runnable notebook cells -- rerun just the PPD-sampling cell to
redo a resample, or just one plot cell to restyle a single figure, without
regenerating everything else.

Uses the shared `src.plotting` library (see `compute_ppds.py` and
`plot_subpop_results.py` at the repo root for the scripted equivalent of this
notebook).

In [ ]:
# -----------------------------------------------------------------------------
# CONFIG -- point this at the run you want, then Run All.
# -----------------------------------------------------------------------------
RESULTS_ROOT = "FinalResults"
FOLDER = "2026-09-14-Final-ThreeComponents-GWTC5-ThreeRegimes-mix-tilt-mmin10-sigpp_low0p5-unconstrained-labels-diagonal-all-mag-rho-fixed-ta090-depth8-1k-combined-3-chains-drawwise-spin-ordered"

N_COMPONENTS = None  # None = infer from result.csv columns (3 or 4)
SEED = 0

# PPD sampling: draws per hyperposterior sample for the main `ppds` table.
OVERSAMPLE = 50
# Monte Carlo samples per hyperposterior draw for the chi_eff/chi_p bands.
# Keep this large -- reducing it reintroduces histogram-binning blockiness in
# broad components (e.g. high spin); generating more samples is the fix, not
# smoothing the bins.
CHI_EFF_CHI_P_SAMPLES = 5000

SKIP_MACROS = False  # the low-spin Bayes-factor macro needs a working Julia/PythonCall.jl setup

In [ ]:
import os
from pathlib import Path

# Keep Julia's lockfiles and precompile cache in the writable project while
# reusing packages already installed in the user's read-only depot. Must be
# set before anything imports juliacall (only the macros cell does).
# The second entry MUST be $HOME/.julia, not a project-local dir -- a
# project-local ".julia" containing only stale compiled/ cache and no
# packages/ fails with "PythonCall.jl did not start properly".
_PROJECT_ROOT = Path(os.getcwd()).resolve()
_JULIA_DEPOT = _PROJECT_ROOT / ".julia-depot"
_JULIA_DEPOT.mkdir(exist_ok=True)
os.environ.setdefault("JULIA_DEPOT_PATH", f"{_JULIA_DEPOT}:{Path.home() / '.julia'}")
os.environ.setdefault("MPLCONFIGDIR", str(_PROJECT_ROOT / ".cache" / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gravpop import load_run

from src.plotting.style import (
    apply_matplotlib_rcparams, component_letters as make_component_letters,
    default_component_colors, default_component_labels,
    APJ_HALF_COLUMN_WIDTH_IN, AASTEX_TEXT_WIDTH_IN,
)
from src.plotting.relabeling import (
    infer_spin_order, relabel_component_hyperparameters, ordered_source_indices_from_letters,
    get_component_ppd_samples, add_chi_eff_chi_p,
)
from src.plotting.ppd_compute import (
    sample_ppds, evaluate_fraction_ppd, fraction_bands_to_frame,
    component_derived_spin_ppd_draws, component_derived_spin_ppd_bands_frame,
    mass_ratio_ppd_bands_frame, mass_contribution_bands_frame, high_spin_tilt_bands_frame,
)
from src.plotting import figures as fig_lib
from src.plotting.macros import assemble_all_macros

apply_matplotlib_rcparams()

result_folder = os.path.join(RESULTS_ROOT, FOLDER)
result_folder

## Load the run and relabel components by physical spin order

In [ ]:
def infer_n_components(df):
    for n in (4, 3):
        if all(f"mu_chi_1_{letter}" in df.columns for letter in "abcd"[:n]):
            return n
    raise ValueError("Could not infer component count (3 or 4) from result.csv columns.")


model_file = Path(result_folder) / "model_and_priors_new.py"
if not model_file.exists():
    model_file = Path(result_folder) / "model_and_priors.py"
run = load_run(str(model_file))

df_raw = pd.read_csv(f"{result_folder}/result.csv")
n_components = N_COMPONENTS or infer_n_components(df_raw)
letters = make_component_letters(n_components)
labels = default_component_labels(n_components)
colors = default_component_colors(n_components)

# Fixed correlation coefficients are intentionally absent from result.csv, but
# GravPop's nested PPD sampler still expects their columns in the input frame.
for component in letters:
    for correlation in (f"rho_chi_{component}", f"rho_cos_tilt_{component}"):
        if correlation not in df_raw.columns:
            df_raw[correlation] = 1e-6

ordered_letters, component_scores = infer_spin_order(df_raw, letters)
ordered_source_indices = ordered_source_indices_from_letters(ordered_letters, letters)
df, _ = relabel_component_hyperparameters(df_raw, ordered_letters, letters)
print(
    "Reordered components by posterior median spin magnitude:",
    ", ".join(
        f"{target.upper()}<-{source} ({component_scores[source]:.3f})"
        for target, source in zip(letters, ordered_letters)
    ),
)

full_model = run["HL"].models[0]
mass_model = full_model.models[0]
spin_model = full_model.models[2]
frac_model = spin_model.mixing_model
df

## Sample PPDs

Rerun this cell alone (with a different `OVERSAMPLE` above) to redraw the
main PPD table without reloading the run.

In [ ]:
ppds = sample_ppds(full_model, spin_model, df_raw, OVERSAMPLE, SEED)
ppds

In [ ]:
component_ppds = add_chi_eff_chi_p(
    get_component_ppd_samples(ppds, component_indices=ordered_source_indices)
)
raw_component_ppds = get_component_ppd_samples(ppds, component_indices=tuple(range(n_components)))

## Fraction ($\eta$) vs $m_1$

In [ ]:
macro_masses = (10, 20, 35, 60)
m1_narrow = np.union1d(np.linspace(3, 100, 100), macro_masses)
m1_wide = np.linspace(3, 200, 200)

w0s_original_labels = evaluate_fraction_ppd(frac_model, df_raw, m1_narrow)
fraction_bands_original = fraction_bands_to_frame(m1_narrow, w0s_original_labels, letters)

w0s_ordered_narrow = w0s_original_labels[:, :, ordered_source_indices]
fraction_bands_narrow = fraction_bands_to_frame(m1_narrow, w0s_ordered_narrow, letters)

w0s_wide = evaluate_fraction_ppd(frac_model, df_raw, m1_wide)[:, :, ordered_source_indices]
fraction_bands_wide = fraction_bands_to_frame(m1_wide, w0s_wide, letters)

In [ ]:
fig = fig_lib.eta_vs_m1_plot(
    fraction_bands_narrow, letters, result_folder, labels, colors,
    output_name="eta_vs_m1.png", figwidth=APJ_HALF_COLUMN_WIDTH_IN, dpi=200,
)

In [ ]:
fig = fig_lib.eta_vs_m1_plot(
    fraction_bands_wide, letters, result_folder, labels, colors,
    output_name="eta_vs_m1-two-column.png", figwidth=AASTEX_TEXT_WIDTH_IN, dpi=200,
)

## Original-label fraction + spin-magnitude summary

In [ ]:
fig = fig_lib.original_label_fraction_spin_magnitude_plot(
    fraction_bands_original, raw_component_ppds, result_folder, letters, labels, colors,
)

## $\chi_{\rm eff}$ / $\chi_p$

In [ ]:
fig = fig_lib.component_chi_eff_chi_p_ppd_plot(
    component_ppds, result_folder, labels, colors,
    scatter=True, output_name="component-by-component-chi-eff-chi-p-PPD.pdf",
)

In [ ]:
fig = fig_lib.component_chi_eff_chi_p_ppd_plot(
    component_ppds, result_folder, labels, colors,
    scatter=False, output_name="component-by-component-chi-eff-chi-p-PPD_kde.pdf",
)

In [ ]:
# Expensive: resamples each component at CHI_EFF_CHI_P_SAMPLES per hyperposterior draw.
grids, density_draws = component_derived_spin_ppd_draws(
    spin_model, df_raw, ordered_source_indices, SEED,
    samples_per_hyperposterior=CHI_EFF_CHI_P_SAMPLES,
)
chi_eff_chi_p_bands = component_derived_spin_ppd_bands_frame(grids, density_draws, letters)

In [ ]:
fig_lib.component_derived_spin_ppd_bands_plot(chi_eff_chi_p_bands, letters, result_folder, labels, colors)

In [ ]:
fig = fig_lib.component_chi_eff_chi_p_filled_composite_plot(
    component_ppds, chi_eff_chi_p_bands, letters, result_folder, labels, colors,
)

## Mass ratio

In [ ]:
fig = fig_lib.mass_ratio_hyperparameter_scatter(df, result_folder, letters, labels, colors)

In [ ]:
try:
    fig = fig_lib.mass_ratio_hyperparameter_kde(df, result_folder, letters, labels, colors)
except ValueError as error:
    if "Contour levels must be increasing" not in str(error):
        raise
    print("Skipping degenerate mass-ratio hyperparameter KDE plot:", error)

In [ ]:
mass_ratio_bands = mass_ratio_ppd_bands_frame(df, letters)
fig = fig_lib.mass_ratio_ppd_band_plot(mass_ratio_bands, letters, result_folder, labels, colors)

## Mass contribution

In [ ]:
mass_contribution_bands = mass_contribution_bands_frame(mass_model, frac_model, df, letters)

fig = fig_lib.component_mass_contribution_ppd_plot(
    mass_contribution_bands, letters, result_folder, labels, colors,
    output_path=f"{result_folder}/component-mass-contributions-PPD-two-column.pdf",
    figsize=(AASTEX_TEXT_WIDTH_IN, APJ_HALF_COLUMN_WIDTH_IN),
)
fig.savefig(f"{result_folder}/component-mass-contributions-PPD.pdf", dpi=400, bbox_inches="tight")

In [ ]:
fig = fig_lib.component_mass_contribution_ppd_plot(
    mass_contribution_bands, letters, result_folder, labels, colors,
    output_path=f"{result_folder}/component-mass-contributions-PPD-one-column.pdf",
    figsize=(APJ_HALF_COLUMN_WIDTH_IN, APJ_HALF_COLUMN_WIDTH_IN),
)

## Component-by-component spin magnitude / tilt (scatter + KDE)

In [ ]:
fig = fig_lib.component_by_component_mag_ppd_plot(
    ppds, result_folder, ordered_source_indices, labels, colors,
    legend_location="upper left", marginals=False,
    figsize=(APJ_HALF_COLUMN_WIDTH_IN, APJ_HALF_COLUMN_WIDTH_IN), dpi=400,
)

In [ ]:
fig = fig_lib.component_by_component_tilt_ppd_plot(
    ppds, result_folder, ordered_source_indices, labels, colors,
    legend_location="upper left", marginals=False,
    figsize=(APJ_HALF_COLUMN_WIDTH_IN, APJ_HALF_COLUMN_WIDTH_IN), dpi=400,
)

In [ ]:
try:
    fig = fig_lib.new_component_by_component_mag_ppd_plot(
        ppds, result_folder, ordered_source_indices, labels, colors,
        legend_location="upper left", marginals=False,
        figsize=(APJ_HALF_COLUMN_WIDTH_IN, APJ_HALF_COLUMN_WIDTH_IN), dpi=400,
    )
except ValueError as error:
    if "Contour levels must be increasing" not in str(error):
        raise
    print("Skipping degenerate component spin-magnitude KDE plot:", error)

In [ ]:
try:
    fig = fig_lib.new_component_by_component_tilt_ppd_plot(
        ppds, result_folder, ordered_source_indices, labels, colors,
        legend_location="upper left", marginals=False,
        figsize=(APJ_HALF_COLUMN_WIDTH_IN, APJ_HALF_COLUMN_WIDTH_IN), dpi=400,
    )
except ValueError as error:
    if "Contour levels must be increasing" not in str(error):
        raise
    print("Skipping degenerate component tilt KDE plot:", error)

## Tilt-mixture $\zeta$

In [ ]:
fig = fig_lib.zeta_cos_tilt_histogram_plot(df, letters, labels, colors, result_folder)
fig

## High-spin $\cos(\theta)$ marginal

In [ ]:
high_spin_tilt_bands = high_spin_tilt_bands_frame(df, high_spin_letter=letters[-1])
fig = fig_lib.high_spin_tilt_marginal_plot(high_spin_tilt_bands, labels[-1], colors, result_folder)

## Spin-hyperparameter comparison

In [ ]:
fig = fig_lib.spin_hyperparameter_comparison_plot(df, letters, labels, result_folder)

## Slow-spin (component A) hyperparameters and PPD

In [ ]:
fig = fig_lib.slow_spin_hyperparameter_corner_plot(df, letters[0], colors[0], result_folder)

In [ ]:
try:
    fig = fig_lib.slow_spin_magnitude_tilt_kde_plot(
        component_ppds[0].sample(10_000, replace=True, random_state=10), result_folder, colors[0],
    )
except ValueError as error:
    if "Contour levels must be increasing" not in str(error):
        raise
    print("Skipping degenerate slow-spin KDE contour plot:", error)

## Mass-break hyperparameter corner

In [ ]:
fig_kde, fig_scatter = fig_lib.mass_break_corner_plots(
    df, colors[1] if len(colors) > 1 else colors[0], result_folder,
)

## Macros

Rerun this cell alone to regenerate `macros.json` (e.g. after fixing the
Julia/PythonCall.jl setup) without redoing any of the plots above.

In [ ]:
import json

all_macros = assemble_all_macros(
    df, ppds, m1_narrow, w0s_ordered_narrow, letters, ordered_source_indices,
    low_spin_letter=letters[0],
)
with open(f"{result_folder}/macros.json", "w", encoding="utf-8") as f:
    json.dump(all_macros, f, indent=2)

all_macros